# DSA 405 Project Milestone 3: Acquisition & Integration
**Author:** Tanmayi Kurra  
**Unity ID:** tkurra  
**Source 2** is one bulk file (Berkeley Lab's public zip, about 241 MB), not a paginated website or an API. How each P3 requirement is handled:

| Requirement | How the code handles it |
| :--- | :--- |
| Rate limited | `polite_get` waits so requests are at least 1 second apart. |
| Honest User-Agent | Every request sends the coursework User-Agent set in the setup cell. |
| Cached | If the verified zip is already in `data/raw/`, no request is made. |
| Pagination | Not applicable: the data is a single file, not paged results. |
| Timeouts and non-200 responses | The request has a timeout; any network error or non-200 status raises `AcquisitionError`. |
| Selectors that match nothing | Not applicable (no HTML parsing). The equivalent risk here is receiving an HTML page instead of the zip, which the code detects. |
| Fails loudly | Any wrong file stops the notebook with an error message instead of producing wrong output silently. |

The raw zip is never modified. `data/raw/MANIFEST_LBNL.json` records its size and SHA-256 so anyone can confirm they have the same file.

In [12]:
# ----------------------------------------------------------------------
# Setup
# ----------------------------------------------------------------------
import hashlib, json, shutil, subprocess, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# Every request identifies who I am and why.
HEADERS = {"User-Agent": "DSA405-student-coursework (NC State; teaching exercise)"}

# Folder layout: raw/ holds untouched downloads only; clean/ holds anything my code derives.
RAW_DIR = Path("data/raw")
PROC_DIR = Path("data/clean")
for d in (RAW_DIR, PROC_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Source 2: Berkeley Lab public data file (facts from my own check on 2026-11-09)
SHORT_URL     = "https://bit.ly/trackingthesun2026"   # link published in LBNL's user guide
ZIP_NAME      = "LBNL_distributed_solar_dataset_public.zip"
CSV_MEMBER    = "TTS_LBNL_public_file_28-Jul-2026_all.csv"
CSV_BYTES     = 2_423_981_133      # uncompressed size, from `unzip -v`
CSV_CRC32     = 0x48D8E4ED         # checksum of the CSV, from `unzip -v`
EXPECTED_ROWS = 3_995_665          # rows counted in the file
MIN_DELAY_S   = 1.0                # one request per second (P3 requirement)

# This file uses Deflate64 compression, which Python's zipfile cannot decompress,
# so I stream it through the system `unzip` command instead.
assert shutil.which("unzip"), "The `unzip` command is required to read this file."
print("pandas", pd.__version__, "| unzip:", shutil.which("unzip"))

pandas 2.3.3 | unzip: /usr/bin/unzip


In [3]:
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser


class AcquisitionError(RuntimeError):
    """Raised when collection cannot be trusted. It is never caught and silenced."""


_last_request = 0.0

def polite_get(url, **kwargs):
    """GET with the honest User-Agent, a timeout, and at most one request per MIN_DELAY_S."""
    global _last_request
    wait = MIN_DELAY_S - (time.monotonic() - _last_request)
    if wait > 0:
        time.sleep(wait)
    _last_request = time.monotonic()
    return requests.get(url, headers=HEADERS, timeout=(10, 60), **kwargs)


def check_robots(url):
    """Print what the site's robots.txt says about this URL. True/False, or None if there are no rules."""
    parts = urlparse(url)
    robots_url = f"{parts.scheme}://{parts.netloc}/robots.txt"
    try:
        r = polite_get(robots_url)
    except requests.RequestException as e:
        raise AcquisitionError(f"Could not fetch {robots_url}: {e}") from e
    print(f"robots.txt: {robots_url} -> HTTP {r.status_code}")
    if r.status_code != 200:
        print("  no usable robots.txt (a missing file is not the same as permission)")
        return None
    rp = RobotFileParser()
    rp.parse(r.text.splitlines())
    allowed = rp.can_fetch(HEADERS["User-Agent"], url)
    print(f"  can_fetch: {allowed} | crawl-delay: {rp.crawl_delay(HEADERS['User-Agent'])}")
    return allowed


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def verify_zip(path):
    """Prove `path` is the real Berkeley Lab zip. Raises AcquisitionError if anything is off."""
    path = Path(path)
    with open(path, "rb") as f:
        if f.read(4) != b"PK\x03\x04":
            raise AcquisitionError(f"{path.name} is not a zip file. It is probably an HTML warning page.")
    try:
        with zipfile.ZipFile(path) as z:          # reads the file list only, no decompression
            info = z.getinfo(CSV_MEMBER)
    except (zipfile.BadZipFile, KeyError) as e:
        raise AcquisitionError(f"{path.name} does not contain {CSV_MEMBER}: {e!r}") from e
    if info.file_size != CSV_BYTES or info.CRC != CSV_CRC32:
        raise AcquisitionError(
            f"CSV inside the zip does not match the version I checked: "
            f"size {info.file_size:,} (expected {CSV_BYTES:,}), CRC {info.CRC:08x} (expected {CSV_CRC32:08x})")
    test = subprocess.run(["unzip", "-tq", str(path)], capture_output=True, text=True)
    if test.returncode != 0:
        raise AcquisitionError(f"unzip integrity test failed: {test.stdout} {test.stderr}")
    sha = sha256_of(path)
    manifest = {"file": path.name, "bytes": path.stat().st_size, "sha256": sha,
                "csv_member": CSV_MEMBER, "csv_bytes": CSV_BYTES, "csv_crc32": f"{CSV_CRC32:08x}",
                "source_short_url": SHORT_URL, "verified_on": time.strftime("%Y-%m-%d")}
    (path.parent / "MANIFEST_LBNL.json").write_text(json.dumps(manifest, indent=2))
    return sha

In [4]:
def acquire_zip(raw_dir=RAW_DIR):
    """Return the verified zip in raw_dir, requesting it only if it is not already there."""
    raw_dir = Path(raw_dir)
    target = raw_dir / ZIP_NAME

    if target.exists():                                    # cache: never fetch twice
        sha = verify_zip(target)
        print(f"Cached file verified, no request made. sha256 starts {sha[:16]}")
        return target

    if check_robots(SHORT_URL) is False:
        raise AcquisitionError("robots.txt disallows this URL for my User-Agent. Stopping.")
    try:
        resp = polite_get(SHORT_URL, stream=True)
    except requests.RequestException as e:
        raise AcquisitionError(f"Request failed (timeout or network error): {e}") from e

    with resp:
        host = urlparse(resp.url).netloc
        ctype = resp.headers.get("Content-Type", "")
        print(f"HTTP {resp.status_code} from {host} | type: {ctype}")
        if resp.status_code != 200:
            raise AcquisitionError(f"HTTP {resp.status_code} from {host}. Not downloading.")
        if check_robots(resp.url) is False:
            raise AcquisitionError(f"robots.txt on {host} disallows this URL. Stopping.")
        if "text/html" in ctype:
            raise AcquisitionError(
                "The server sent an HTML page, not the zip file (for a file this large, Google Drive "
                "shows a 'can't scan for viruses' page). Download the file once by hand and put it in "
                f"{raw_dir}/ as {ZIP_NAME}, then run this cell again.")

        raw_dir.mkdir(parents=True, exist_ok=True)
        part = target.with_suffix(".part")
        received = 0
        try:
            with open(part, "wb") as out:
                for block in resp.iter_content(chunk_size=1 << 20):
                    out.write(block)
                    received += len(block)
            expected = resp.headers.get("Content-Length")
            if expected and received != int(expected):
                raise AcquisitionError(f"Download cut short: got {received:,} of {int(expected):,} bytes.")
            sha = verify_zip(part)
            part.rename(target)
        except requests.RequestException as e:
            raise AcquisitionError(f"Download interrupted: {e}") from e
        finally:
            if part.exists():
                part.unlink()                              # never leave a half-finished file
    print(f"Downloaded and verified: {received:,} bytes, sha256 starts {sha[:16]}")
    return target

In [5]:
# Loud-failure demonstration. With nothing cached, the request returns a warning page instead of
# the zip. The code must stop with a clear message and must not continue with bad data.
import tempfile
with tempfile.TemporaryDirectory() as empty_dir:
    try:
        acquire_zip(empty_dir)
        print("Unexpected: the download worked without the warning page.")
    except AcquisitionError as err:
        print("CAUGHT, failed loudly:", err)

robots.txt: https://bit.ly/robots.txt -> HTTP 200
  can_fetch: True | crawl-delay: None
HTTP 200 from drive.google.com | type: text/html; charset=utf-8
robots.txt: https://drive.google.com/robots.txt -> HTTP 200
  can_fetch: True | crawl-delay: 1
CAUGHT, failed loudly: The server sent an HTML page, not the zip file (for a file this large, Google Drive shows a 'can't scan for viruses' page). Download the file once by hand and put it in /var/folders/mj/f3f58n5s4cbgpp72tfrcl3340000gn/T/tmp1m2kg1it/ as LBNL_distributed_solar_dataset_public.zip, then run this cell again.


In [9]:
ZIP_PATH = acquire_zip()
print(ZIP_PATH, f"{ZIP_PATH.stat().st_size:,} bytes")

Cached file verified, no request made. sha256 starts 75f3b4a5cdf6c018
data/raw/LBNL_distributed_solar_dataset_public.zip 252,277,242 bytes


In [8]:
shutil.copy(Path.home() / "Downloads" / ZIP_NAME, RAW_DIR / ZIP_NAME)

PosixPath('data/raw/LBNL_distributed_solar_dataset_public.zip')

## Part 2: Ethics and legality in practice

### Does the final version match the A8 plan?

| A8 plan | What I actually did | Match? |
|---|---|---|
| A8 covered only the EIA API: robots.txt, terms of use, public-domain license, API key kept out of the code, 2-second pauses. | EIA was collected and cleaned in P2 under that plan. | Yes |
| A8 had no plan for Berkeley Lab, the source collected in this notebook. | I collected it here without a prior written plan. For this source I used the P3 minimum of one request per second. Google's robots.txt crawl-delay is also 1 second. | No (drift 1) |

### Drift: where the final version differs from the plan, and why

1. **No A8 plan for the second source.** A8 covered only EIA. This section is the plan I should have written for Berkeley Lab, applied in the code: robots.txt is checked by the code, the license is recorded below, and the User-Agent is stated.
2. **The source is a bulk file, not a scraped page or an API.** There are no selectors, pagination or rate-limit headers to read. The equivalent risks are receiving an HTML page instead of the file, and a truncated or changed file. The code verifies file type, size, CRC-32 and zip integrity, and raises an error if any check fails.
3. **I downloaded the file by hand once.** The short link leads to a Google Drive file over 100 MB, and Drive shows a "can't scan for viruses" HTML page before the real file. My code does not click through that page. I downloaded the file once in my browser using the "Download anyway" button that Drive shows every visitor. After that, the code reads the cached file and verifies it.
4. **The file is 2.4 GB and uses Deflate64 compression.** Python's `zipfile` cannot read it, so the zip is not committed to GitHub and the CSV is streamed through the system `unzip` command. The repo holds only a small state-year summary in `data/clean/` and a manifest with the zip's size and SHA-256.
5. **The public file has fewer systems than P1 said.** P1 cited about 5.3 million systems (the full Berkeley Lab database). The public file has 3,995,665 systems, because confidential records are excluded.

### Guardrails re-confirmed

- **No login.** I did not sign in to any account to get the file.
- **No paywall.** The file is free and public.
- **No personal data.** The records are project-level, one row per installed system, with no customer names. I load only four columns (`state`, `installation_date`, `PV_system_size_DC`, `technology_type`) and never load `zip_code` or `city`.
- **No circumvention.** The code never bypasses the Drive warning page, retries around a 403, or hides its identity. If a protection stops the download, the code raises an error and stops.
- **Honest User-Agent:** `DSA405-student-coursework (NC State; teaching exercise)`.
- **robots.txt**, checked by the code at run time:
  - bit.ly: HTTP 200, my User-Agent is allowed, no crawl-delay.
  - drive.google.com: HTTP 200, my User-Agent is allowed, crawl-delay 1 second, which my one-request-per-second spacing meets.
- **License and terms:** Berkeley Lab's project page for this dataset states no license. The Data.gov catalog record for Tracking the Sun, published by Lawrence Berkeley National Laboratory, lists CC BY 4.0. I used the data only for this coursework and credit Lawrence Berkeley National Laboratory as the source.
- **Rate and volume:** one request per second, about 3 requests on the first run and none on re-runs.


'N/A'  (107 records)
  types : {'Sign': 55, 'Electrical': 52}
  months: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']

0  (23 records)
  types : {'Demolition': 23}
  months: ['01', '02', '03', '05', '06', '07', '08', '09', '10', '11', '12']

-1  (18 records)
  types : {'Commercial Upfit': 4, 'Residential Alteration': 3, 'New Single Family': 3}
  months: ['03', '04']

None  (34 records)
  types : {'Plumbing': 8, 'New Single Family': 8, 'Mechanical': 6}
  months: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']

''  (23 records)
  types : {'Mechanical': 7, 'Residential Alteration': 4, 'Commercial Upfit': 4}
  months: ['01', '02', '03', '04', '05', '06', '07', '08', '10', '11', '12']


In [13]:
import json, subprocess
import pandas as pd

USECOLS    = ["state", "installation_date", "PV_system_size_DC", "technology_type"]
FIRST_YEAR_OK, LAST_YEAR = 1990, 2025          # EIA's last year is 2025
EIA_FIRST_YEAR = 2001                          # the running total must start no later than this
CAP_CSV  = PROC_DIR / "tts_state_year_capacity.csv"
ACCT_JSON = PROC_DIR / "tts_accounting.json"

def read_tts(zip_path, chunksize=500_000):
    """Stream the CSV out of the zip through `unzip -p` (Python's zipfile cannot read Deflate64)."""
    proc = subprocess.Popen(["unzip", "-p", str(zip_path), CSV_MEMBER],
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    try:
        for chunk in pd.read_csv(proc.stdout, usecols=USECOLS, chunksize=chunksize,
                                 dtype={"state": str, "technology_type": str, "installation_date": str}):
            yield chunk
    finally:
        proc.stdout.close()
    err = proc.stderr.read().decode(errors="replace")
    if proc.wait() != 0:
        raise AcquisitionError(f"unzip failed while streaming the CSV: {err}")

def aggregate_tts(zip_path):
    acct = dict(rows_read=0, dropped_storage_only=0, dropped_bad_state=0,
                dropped_bad_size=0, dropped_outside_years=0, kept=0, kept_tech_redacted=0)
    parts = []
    for ch in read_tts(zip_path):
        acct["rows_read"] += len(ch)
        tech  = ch["technology_type"].astype(str).str.strip().str.lower()
        state = ch["state"].astype(str).str.strip().str.upper()
        size  = pd.to_numeric(ch["PV_system_size_DC"], errors="coerce")
        year  = pd.to_numeric(ch["installation_date"].astype(str).str[:4], errors="coerce")

        # Each row falls in exactly one bucket, so the counts must add up to rows_read.
        storage   = tech.eq("storage-only")
        bad_state = ~storage & ~state.str.fullmatch(r"[A-Z]{2}")
        bad_size  = ~storage & ~bad_state & ~(size > 0)          # catches -1, 0 and NaN
        bad_year  = ~storage & ~bad_state & ~bad_size & ~year.between(FIRST_YEAR_OK, LAST_YEAR)
        keep      = ~(storage | bad_state | bad_size | bad_year)

        acct["dropped_storage_only"]   += int(storage.sum())
        acct["dropped_bad_state"]      += int(bad_state.sum())
        acct["dropped_bad_size"]       += int(bad_size.sum())
        acct["dropped_outside_years"]  += int(bad_year.sum())
        acct["kept"]                   += int(keep.sum())
        acct["kept_tech_redacted"]     += int((keep & tech.eq("redacted")).sum())

        parts.append(pd.DataFrame({"state": state[keep], "year": year[keep].astype(int),
                                   "kw": size[keep]})
                     .groupby(["state", "year"], as_index=False)
                     .agg(kw_added=("kw", "sum"), n_added=("kw", "size")))

    if acct["rows_read"] != EXPECTED_ROWS:
        raise AcquisitionError(f"Read {acct['rows_read']:,} rows, expected {EXPECTED_ROWS:,}. "
                               "The file is not the version this notebook was written for.")
    dropped = sum(v for k, v in acct.items() if k.startswith("dropped_"))
    if acct["kept"] + dropped != acct["rows_read"]:
        raise AcquisitionError(f"Row accounting does not add up: {acct}")

    added = pd.concat(parts).groupby(["state", "year"], as_index=False).sum()
    # A state-year with no installs is a real zero, so build the full grid before the running total.
    grid = pd.MultiIndex.from_product([sorted(added["state"].unique()),
                                       range(min(int(added["year"].min()), EIA_FIRST_YEAR), LAST_YEAR + 1)],
                                      names=["state", "year"])
    cap = added.set_index(["state", "year"]).reindex(grid, fill_value=0).reset_index()
    cap["capacity_kw_cum"] = cap.groupby("state")["kw_added"].cumsum()
    cap["systems_cum"]     = cap.groupby("state")["n_added"].cumsum()
    return cap, acct

if CAP_CSV.exists() and ACCT_JSON.exists():
    cap = pd.read_csv(CAP_CSV)
    acct = json.loads(ACCT_JSON.read_text())
    print("Loaded cached state-year summary (delete data/clean/tts_* to recompute).")
else:
    cap, acct = aggregate_tts(ZIP_PATH)
    cap.to_csv(CAP_CSV, index=False)
    ACCT_JSON.write_text(json.dumps(acct, indent=2))
print(json.dumps(acct, indent=2))
print(cap.shape, "| states:", cap["state"].nunique(), "| years:", cap["year"].min(), "-", cap["year"].max())

{
  "rows_read": 3995665,
  "dropped_storage_only": 52922,
  "dropped_bad_state": 0,
  "dropped_bad_size": 4913,
  "dropped_outside_years": 0,
  "kept": 3937830,
  "kept_tech_redacted": 230106
}
(756, 6) | states: 27 | years: 1998 - 2025


## Part 3: Join design

| | |
|---|---|
| **Join keys** | `(state_id, year)` in both tables |
| **Grain, left table (EIA)** | one row per state per year: average residential price (cents/kWh) |
| **Grain, right table (Berkeley Lab, after aggregation)** | one row per state per year: cumulative installed solar (kW DC) |
| **Relationship** | one-to-one |
| **Expected rows after the join** | 1,275 (51 states and DC × 25 years, the same as the EIA table) |
| **Join type and why** | **Left, from EIA.** The question is about price, so every state-year with a price must stay. An inner join would drop every state the public Berkeley Lab file does not cover 600 price rows, which is 24 of the 51 jurisdictions and make the result look national when it is not. A right or outer join would add territory rows and years with no price. |
| **Unmatched rows** | Kept with `NA` capacity, not set to 0. "Not in this dataset" is different from "has zero solar". |

**Which failure does my code guard against?** Both. The left join guards against the inner-join "wreck" (losing price rows). Aggregating the 4 million system rows to one row per state-year before joining, together with `validate="one_to_one"`, guards against fan-out (system-level rows multiplying the price rows).

In [15]:
EIA_URL = "https://raw.githubusercontent.com/tiakurra/dsa405-project/main/data/clean/eia_retail_sales_clean.csv"
eia = pd.read_csv(EIA_URL)
assert len(eia) == 1_275, "EIA table is not the 1,275-row file produced in P2"
cap_join = cap.rename(columns={"state": "state_id"})[["state_id", "year", "capacity_kw_cum", "systems_cum"]]

# Territories are in the solar file but not in the EIA state table; leaving them out is expected and is reported here.
TERRITORIES = {"PR", "VI", "GU", "AS", "MP"}
# Keys on the right side that the left side does not know about (a left join would silently drop them):
only_right = sorted(set(cap_join["state_id"]) - set(eia["state_id"]))
print("TTS states not in the EIA table:", only_right or "none")

N_EXPECTED = len(eia)
merged = eia.merge(cap_join, how="left", on=["state_id", "year"],
                   validate="one_to_one", indicator=True)
print(merged.shape, merged["_merge"].value_counts().to_dict())

TTS states not in the EIA table: none
(1275, 10) {'both': 675, 'left_only': 600, 'right_only': 0}


In [16]:
covered = merged.groupby("state_id")["capacity_kw_cum"].apply(lambda s: s.notna().all())
print("Jurisdictions with solar data:", int(covered.sum()), sorted(covered[covered].index))
print("Without solar data:", int((~covered).sum()),
      "| price rows an inner join would have dropped:", int(merged["capacity_kw_cum"].isna().sum()))
installs = cap.groupby("year")["n_added"].sum()
print(installs.loc[2015:2025].to_string())
share = cap.groupby("state")["n_added"].sum().sort_values(ascending=False)
print((share / share.sum()).head(5).round(3).to_string())

Jurisdictions with solar data: 27 ['AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'IL', 'MA', 'MD', 'ME', 'MN', 'NH', 'NJ', 'NM', 'NY', 'OH', 'OR', 'PA', 'RI', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI']
Without solar data: 24 | price rows an inner join would have dropped: 600
year
2015    257518
2016    297941
2017    238219
2018    243723
2019    267401
2020    279318
2021    339562
2022    442772
2023    483895
2024    295483
2025    259320
state
CA    0.563
AZ    0.085
TX    0.062
NJ    0.058
NY    0.046


In [17]:
def verify(df, n_expected, acct, only_right):
    """Each check states an assumption about the data. If it stops being true, this reports it."""
    fails = []
    # protects against: a fan-out join multiplying rows, or an inner join silently dropping them
    if len(df) != n_expected:
        fails.append(f"row count {len(df)} != expected {n_expected}")
    # protects against: duplicate (state, year) rows that would double-count a state in the scatter plot
    if df.duplicated(["state_id", "year"]).any():
        fails.append("duplicate (state_id, year) keys")
    # protects against: a wrong grain; every state should have exactly one row per year, 25 years
    per_state = df.groupby("state_id").size()
    if not (per_state == df["year"].nunique()).all():
        fails.append("some state does not have one row per year")
    # protects against: the -1 'missing' code being summed as if it were a real kW value
    if (df["capacity_kw_cum"].dropna() < 0).any():
        fails.append("negative capacity (a -1 sentinel was summed)")
    # protects against: impossible prices (cents/kWh) from a unit or parsing error
    if not df["price_cents_kwh"].between(1, 100).all():
        fails.append("price outside 1-100 cents/kWh")
    # protects against: cumulative capacity going DOWN over time, which means rows were lost or mis-dated
    cov = df.dropna(subset=["capacity_kw_cum"]).sort_values(["state_id", "year"])
    if (cov.groupby("state_id")["capacity_kw_cum"].diff().dropna() < 0).any():
        fails.append("cumulative capacity decreases within a state")
    # protects against: half-matched states; a state should have capacity for all years or for none
    na_by_state = df.groupby("state_id")["capacity_kw_cum"].apply(lambda s: s.isna().mean())
    if not na_by_state.isin([0.0, 1.0]).all():
        fails.append("a state is matched in some years but not others")
    # protects against: a real state in the solar file that the EIA table lacks (its rows would vanish from a left join)
    if set(only_right) - TERRITORIES:
        fails.append(f"solar-file states missing from EIA table: {sorted(set(only_right) - TERRITORIES)}")
    # protects against: rows silently dropped during aggregation without being counted
    dropped = sum(v for k, v in acct.items() if k.startswith("dropped_"))
    if acct["kept"] + dropped != acct["rows_read"]:
        fails.append("aggregation row accounting does not reconcile")
    return fails

failures = verify(merged, N_EXPECTED, acct, only_right)
assert not failures, failures
print("All checks passed.")

All checks passed.


### The verification suite checks

The nine checks above each state an assumption and what it protects against. They cover row count, key uniqueness, grain (25 rows per state), the `-1` missing-value code, price range, cumulative capacity never decreasing, all-or-nothing matching per state, solar-file states missing from EIA, and the aggregation row accounting.

**Result on the real data: all checks passed.** The merged table has 1,275 rows, with 675 matched and 600 unmatched (kept as `NA`).

## Part 5: One thing that surprised me

I expected the Berkeley Lab file to cover the whole country. It does not. It has solar systems in only 27 of the 51 jurisdictions in my EIA table (26 states plus DC), so 24 jurisdictions, including North Carolina, Nevada and Hawaii, have no solar data at all. That is 600 of my 1,275 price rows (24 × 25 years). An inner join would have deleted them silently and left a sample that looked national but was not. This is why I used a left join and kept the missing capacity as `NA` instead of 0.

The covered states are also lopsided. California alone has 56.3% of the systems, followed by Arizona (8.5%), Texas (6.2%), New Jersey (5.8%) and New York (4.6%). So any result will be driven mostly by a few states, and it cannot speak for the country.

A second surprise is in the most recent years. The number of systems installed per year rises to 483,895 in 2023, then falls to 295,483 in 2024 and 259,320 in 2025, about 39% and 46% below 2023. I do not know why. It could be reporting delay in the public file, but I have not confirmed that. Because my capacity measure is a running total of installs, the 2024 and 2025 totals may be understated, so when I compare price and capacity for the most recent year I will also check 2023.

### 5. AI Statement

I used Claude to help me code and understand the code. I also used it to come up with explanations as well as go through the rubric to see how to do better and fix my work.


## Project Self-Scored Rubric

| Criterion | Weight | Score | Justification |
|---|---|---|---|
| 1. Acquisition code | ×2 | 4 | Requests are limited to one per second with an honest User-Agent. A cache check means a re-run makes no request. Timeouts, non-200 responses and an HTML page instead of the zip are all handled, and pagination and selectors are explained as not applicable. The file's type, size, CRC-32 and zip integrity are verified, and a manifest records its SHA-256. One failure (the Drive warning page) is demonstrated and caught with "failed loudly". |
| 2. Ethics & legality in practice | ×2 | 3 | The A8 comparison table and five drift items (with reasons) are written, and all four guardrails are re-confirmed, with the robots.txt output and license information recorded. I lose a point because A8 had no plan for this source, and Berkeley Lab's own page states no license (the CC BY 4.0 listing comes from the Data.gov catalog record). |
| 3. Join design | ×2 | 4 | Keys, both grains, relationship and the expected row count (1,275) are stated before the merge. A left join is chosen over inner, right and outer with reasons, and unmatched rows are kept as `NA`. The merge uses `validate="one_to_one"`, and the notebook says which failure (inner-join row loss and fan-out) it guards against. |
| 4. Verification suite | ×3 | 4 | Nine checks, each with a "protects against" comment, written as a function that returns a list of failures. They cover row count, key uniqueness, grain, the `-1` code, price range, monotonic capacity, all-or-nothing matching, missing states and row accounting. All pass on the real data. |
| 5. Reproducibility & readability | ×1 | 3 | The notebook runs top to bottom from the repo, with the zip cached and verified by size, CRC and SHA-256, and the README is updated. I lose a point because the 241 MB zip is not in the repo, so a reader must download it by hand first. |